# Módulo F1 · Fundamentos de Python para datos

**Caso:** eres analista en *TiendaSur*, una cadena con 5 tiendas. Dirección quiere saber qué tiendas, categorías y meses generan más ingresos, y si las promociones funcionan.

En este notebook aprenderás a:
1. Usar las estructuras básicas de Python.
2. Trabajar con arrays de NumPy.
3. Cargar, limpiar, agrupar y combinar datos con pandas.
4. Visualizar resultados con matplotlib y seaborn.

> Ejecuta las celdas en orden con `Shift + Enter`. Las celdas marcadas como **Ejercicio** son para ti.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 2)

## 1. Python básico en 5 minutos

In [ ]:
# Tipos básicos
precio = 19.99          # float
unidades = 3            # int
producto = "Café"       # str
en_oferta = True        # bool

total = precio * unidades
print(f"{unidades} x {producto} = {total:.2f} €")

# Listas y diccionarios
ventas_semana = [120, 135, 98, 160, 210, 250, 190]
tienda = {"nombre": "Centro", "ciudad": "Sevilla", "m2": 450}

print("Media semanal:", sum(ventas_semana) / len(ventas_semana))
print("Ciudad:", tienda["ciudad"])

In [ ]:
# Funciones y comprensiones de listas
def aplicar_descuento(precio, porcentaje=10):
    """Devuelve el precio tras aplicar un descuento en %."""
    return precio * (1 - porcentaje / 100)

precios = [10, 25, 40]
print([aplicar_descuento(p) for p in precios])
print([aplicar_descuento(p, 25) for p in precios if p > 20])

## 2. NumPy: cálculo vectorizado

NumPy opera sobre arrays completos sin bucles explícitos: es más rápido y más legible.

In [ ]:
v = np.array(ventas_semana)
print("Media:", v.mean(), "| Desv. típica:", v.std().round(1))
print("Días por encima de la media:", (v > v.mean()).sum())
print("Crecimiento diario (%):", (np.diff(v) / v[:-1] * 100).round(1))

## 3. Generamos los datos de TiendaSur

En un proyecto real harías `pd.read_csv("ventas.csv")` o leerías de una base de datos. Aquí simulamos un año de ventas para que el notebook funcione sin archivos externos.

In [ ]:
rng = np.random.default_rng(42)

tiendas = pd.DataFrame({
    "tienda_id": [1, 2, 3, 4, 5],
    "tienda": ["Centro", "Norte", "Sur", "Puerto", "Aeropuerto"],
    "ciudad": ["Sevilla", "Sevilla", "Cádiz", "Cádiz", "Málaga"],
    "m2": [450, 300, 380, 220, 150],
})

categorias = {"Alimentación": 8, "Bebidas": 5, "Hogar": 15, "Electrónica": 60, "Ropa": 25}
fechas = pd.date_range("2025-01-01", "2025-12-31", freq="D")

n = 20_000
ventas = pd.DataFrame({
    "fecha": rng.choice(fechas, n),
    "tienda_id": rng.choice(tiendas["tienda_id"], n, p=[0.3, 0.2, 0.2, 0.15, 0.15]),
    "categoria": rng.choice(list(categorias), n, p=[0.35, 0.25, 0.15, 0.1, 0.15]),
    "promocion": rng.random(n) < 0.2,
})
precio_base = ventas["categoria"].map(categorias)
estacionalidad = 1 + 0.3 * np.sin(2 * np.pi * (ventas["fecha"].dt.dayofyear - 80) / 365)
ventas["unidades"] = rng.poisson(2 * estacionalidad * np.where(ventas["promocion"], 1.4, 1.0)) + 1
ventas["precio_unitario"] = (precio_base * rng.normal(1, 0.1, n) * np.where(ventas["promocion"], 0.85, 1)).round(2)

# Introducimos problemas de calidad típicos
ventas.loc[rng.choice(n, 150, replace=False), "precio_unitario"] = np.nan
ventas = pd.concat([ventas, ventas.sample(80, random_state=1)])  # duplicados

ventas.head()

## 4. Primer vistazo y limpieza

In [ ]:
ventas.info()

In [ ]:
print("Valores nulos por columna:\n", ventas.isna().sum(), sep="")
print("\nFilas duplicadas:", ventas.duplicated().sum())

In [ ]:
ventas = ventas.drop_duplicates()
# Imputamos el precio que falta con la mediana de su categoría
ventas["precio_unitario"] = ventas["precio_unitario"].fillna(
    ventas.groupby("categoria")["precio_unitario"].transform("median")
)
ventas["ingreso"] = ventas["unidades"] * ventas["precio_unitario"]
ventas["mes"] = ventas["fecha"].dt.to_period("M")
print("Filas tras limpiar:", len(ventas), "| Nulos restantes:", ventas.isna().sum().sum())

## 5. Preguntas de negocio con `groupby` y `merge`

In [ ]:
# ¿Qué tienda ingresa más? ¿Y por metro cuadrado?
por_tienda = (
    ventas.groupby("tienda_id", as_index=False)["ingreso"].sum()
    .merge(tiendas, on="tienda_id")
    .assign(ingreso_m2=lambda d: d["ingreso"] / d["m2"])
    .sort_values("ingreso", ascending=False)
)
por_tienda[["tienda", "ciudad", "ingreso", "ingreso_m2"]]

In [ ]:
# Ingresos por categoría y tienda (tabla dinámica)
pd.pivot_table(
    ventas.merge(tiendas, on="tienda_id"),
    index="categoria", columns="tienda", values="ingreso", aggfunc="sum",
).round(0)

In [ ]:
# ¿Las promociones aumentan las unidades por ticket?
ventas.groupby("promocion")[["unidades", "precio_unitario", "ingreso"]].mean()

Las promociones elevan las unidades por ticket, pero el precio baja. Que el **ingreso** medio suba o no depende de ambos efectos: es justo el tipo de pregunta que formalizaremos con pruebas de hipótesis en el módulo 04.

## 6. Visualización

In [ ]:
mensual = ventas.groupby("mes")["ingreso"].sum()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

mensual.plot(ax=axes[0], marker="o")
axes[0].set(title="Ingresos mensuales", xlabel="", ylabel="€")

sns.barplot(data=por_tienda, x="tienda", y="ingreso_m2", ax=axes[1], color="steelblue")
axes[1].set(title="Ingreso por m² y tienda", xlabel="", ylabel="€/m²")
plt.tight_layout()
plt.show()

In [ ]:
sns.boxplot(data=ventas, x="categoria", y="ingreso", hue="promocion", showfliers=False)
plt.title("Distribución del ingreso por ticket")
plt.show()

## Ejercicios

1. ¿Qué día de la semana se vende más? Pista: `ventas["fecha"].dt.day_name()`.
2. Calcula la cuota (%) de cada categoría sobre el ingreso total de cada ciudad.
3. ¿Qué mes tiene el mayor ingreso en la tienda *Aeropuerto*? ¿Coincide con el del resto?
4. Crea una columna `ticket_grande` que valga `True` si el ingreso supera el percentil 90 y calcula qué % de tickets grandes ocurre en promoción.
5. Haz un gráfico que ayude a Dirección a decidir en qué tienda abrir una sección de Electrónica más grande. Justifica tu elección en una frase.

In [ ]:
# Ejercicio 1 — TU CÓDIGO AQUÍ

In [ ]:
# Ejercicio 2 — TU CÓDIGO AQUÍ